In [1]:
import sys
import os
import importlib
import anndata as ad
import pandas as pd
import scanpy as sc
import squidpy as sq
import numpy as np

/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/squidpy/gr/_utils.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  CAN_USE_SPARSE_ARRAY = Version(anndata.

In [2]:
sys.path.append(os.path.abspath("/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/"))
import helpers
importlib.reload(helpers)
import tcell_classifier
importlib.reload(tcell_classifier)

<module 'tcell_classifier' from '/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/tcell_classifier.py'>

In [3]:
data_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Cosmx6k/processed_data/"
res_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Cosmx6k/results/final/1_nico_nextseq/"
if not (os.path.exists(res_dir)):
    os.makedirs(res_dir)

In [4]:
dat = pd.read_excel(data_dir+"NICO_NextSeq.xlsx")
metadata = dat[['SampleID', 'TumorType', 'Patient']]
dat = dat.drop(columns=['SampleID', 'TumorType', 'Patient'])

keep_genes = (dat >= 5).sum(axis=0) >= 20
dat = dat.loc[:, keep_genes]

pre_primary_samples = metadata[metadata['TumorType'] == 'Pre-NIVO Primary'].index
dat_pre_primary = dat.loc[pre_primary_samples]
metadata_pre_primary = metadata.loc[pre_primary_samples]
post_primary_samples = metadata[metadata['TumorType'] == 'Post-NIVO Primary'].index
dat_post_primary = dat.loc[post_primary_samples]
metadata_post_primary = metadata.loc[post_primary_samples]
pre_patient = dat_pre_primary.groupby(metadata_pre_primary.loc[pre_primary_samples, 'Patient']).mean()
post_patient = dat_post_primary.groupby(metadata_post_primary.loc[post_primary_samples, 'Patient']).mean()
log2fc = np.log2(post_patient + 1) - np.log2(pre_patient + 1)

In [5]:
from sklearn.preprocessing import StandardScaler
log2fc_scaled = pd.DataFrame(StandardScaler().fit_transform(log2fc),
                           columns=log2fc.columns,
                           index=log2fc.index)

In [6]:
adata = ad.AnnData(
    X=log2fc_scaled.values,                        # rows = patients, columns = genes
    obs=pd.DataFrame(index=log2fc_scaled.index),   # patient IDs
    var=pd.DataFrame(index=log2fc_scaled.columns))
adata.obs['Patient'] = adata.obs.index.astype('category')
adata.obs.index.names = ['pat_names']

In [7]:
adata_bc = helpers.batch_correction(adata, "nobc", "", npcs = 10, dis_met = "euclidean")

2026-07-29 16:18:39 - INFO - ### Batch Correction ###
2026-07-29 16:18:39 - INFO - ## Method = nobc, Batch factor = 
2026-07-29 16:18:39 - INFO - ### No Batch Correction NPC: npcs ###
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


10 18 1396


In [18]:
adata_bc.write_h5ad(res_dir + "bulk_adata.h5ad")